In [1]:
import os

data_path = "../data/threat_intel"

print(os.listdir(data_path))

['botnet.txt', 'ddos_attack.txt', 'phishing.txt', 'ransomware.txt', 'sql_injection.txt']


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [3]:
loader = TextLoader(
    r"C:\Users\Abhishek N Salian\Desktop\cyber-ai-platform\data\threat_intel\ddos_attack.txt"
)

documents = loader.load()

documents

[Document(metadata={'source': 'C:\\Users\\Abhishek N Salian\\Desktop\\cyber-ai-platform\\data\\threat_intel\\ddos_attack.txt'}, page_content='A Distributed Denial of Service (DDoS) attack is a cyberattack in which multiple compromised systems flood a target server, network, or service with traffic.\n\nThe goal is to exhaust resources and make the service unavailable to legitimate users.\n\nCommon indicators:\n- Extremely high traffic volume\n- Large number of requests from multiple IPs\n- Network congestion\n- Service slowdown or outage\n\nMITRE ATT&CK Technique:\n- T1498: Network Denial of Service\n\nCommon mitigation strategies:\n- Rate limiting\n- Traffic filtering\n- Web Application Firewall (WAF)\n- Load balancing\n- Intrusion Detection Systems (IDS)\n\nDDoS attacks are commonly used against:\n- websites\n- cloud infrastructure\n- gaming servers\n- financial systems')]

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print("Number of chunks:", len(docs))

Number of chunks: 3


In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Abhishek N Salian\AppData\Local\Temp\ipykernel_2808\2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
vector_db = Chroma.from_documents(
    docs,
    embedding_model,
    persist_directory="../rag/chroma_db"
)

In [7]:
query = "How can DDoS attacks be mitigated?"

results = vector_db.similarity_search(query, k=2)

results

[Document(metadata={'source': 'C:\\Users\\Abhishek N Salian\\Desktop\\cyber-ai-platform\\data\\threat_intel\\ddos_attack.txt'}, page_content='A Distributed Denial of Service (DDoS) attack is a cyberattack in which multiple compromised systems flood a target server, network, or service with traffic.\n\nThe goal is to exhaust resources and make the service unavailable to legitimate users.'),
 Document(metadata={'source': 'C:\\Users\\Abhishek N Salian\\Desktop\\cyber-ai-platform\\data\\threat_intel\\ddos_attack.txt'}, page_content='Common mitigation strategies:\n- Rate limiting\n- Traffic filtering\n- Web Application Firewall (WAF)\n- Load balancing\n- Intrusion Detection Systems (IDS)\n\nDDoS attacks are commonly used against:\n- websites\n- cloud infrastructure\n- gaming servers\n- financial systems')]

In [8]:
predicted_attack = "DDoS"

query = f"""
What is {predicted_attack} attack and how can it be mitigated?
"""

results = vector_db.similarity_search(query, k=2)

print("=== AI Threat Intelligence Report ===\n")

print("Detected Threat:", predicted_attack)

print("\nRelevant Intelligence:\n")

for result in results:
    print(result.page_content)
    print("\n-----------------\n")

=== AI Threat Intelligence Report ===

Detected Threat: DDoS

Relevant Intelligence:

A Distributed Denial of Service (DDoS) attack is a cyberattack in which multiple compromised systems flood a target server, network, or service with traffic.

The goal is to exhaust resources and make the service unavailable to legitimate users.

-----------------

Common mitigation strategies:
- Rate limiting
- Traffic filtering
- Web Application Firewall (WAF)
- Load balancing
- Intrusion Detection Systems (IDS)

DDoS attacks are commonly used against:
- websites
- cloud infrastructure
- gaming servers
- financial systems

-----------------



In [9]:
from pathlib import Path

threat_path = Path(
    r"C:\Users\Abhishek N Salian\Desktop\cyber-ai-platform\data\threat_intel"
)

all_documents = []

for file in threat_path.glob("*.txt"):
    loader = TextLoader(str(file))
    docs = loader.load()
    all_documents.extend(docs)

print("Total documents loaded:", len(all_documents))

Total documents loaded: 5


In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(all_documents)

print("Total chunks:", len(split_docs))

Total chunks: 14


In [11]:
multi_vector_db = Chroma.from_documents(
    split_docs,
    embedding_model,
    persist_directory="../rag/multi_threat_db"
)

In [12]:
query = "How does ransomware spread?"

results = multi_vector_db.similarity_search(query, k=2)

for i, result in enumerate(results):
    print(f"\nResult {i+1}:\n")
    print(result.page_content)


Result 1:

Ransomware is malware that encrypts victim files and demands payment for decryption.

Attackers commonly spread ransomware through:
- phishing emails
- malicious downloads
- exploit kits
- remote desktop attacks

Result 2:

Ransomware is malware that encrypts victim files and demands payment for decryption.

Attackers commonly spread ransomware through:
- phishing emails
- malicious downloads
- exploit kits
- remote desktop attacks


In [13]:
import ollama

In [14]:
query = "How can ransomware attacks be mitigated?"

results = multi_vector_db.similarity_search(query, k=2)

context = "\n".join([doc.page_content for doc in results])

prompt = f"""
You are a cybersecurity analyst.

Using the threat intelligence context below, generate a professional cybersecurity report.

Threat Intelligence Context:
{context}

User Question:
{query}

Generate:
1. Threat Summary
2. MITRE ATT&CK Mapping
3. Mitigation Recommendations
4. Potential Targets
"""

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(response['message']['content'])

**Threat Intelligence Report: Ransomware Attack Mitigation**

**Threat Summary**

This report provides an overview of the ransomware threat and offers recommendations for mitigating these attacks.

Ransomware is a type of malware that encrypts victim files, demanding payment in exchange for decryption. The attackers commonly spread ransomware through phishing emails, malicious downloads, exploit kits, and remote desktop attacks.

**MITRE ATT&CK Mapping**

The following MITRE ATT&CK mapping illustrates the tactics, techniques, and procedures (TTPs) used by attackers to spread ransomware:

* **Initial Lateral Movement**: Attackers use phishing emails or exploited vulnerabilities in software to gain initial access.
	+ TTP: Phishing Email
	+ TTP: Exploit Kit
* **Lateral Movement**: Once inside the network, attackers move laterally to infect other systems and devices.
	+ TTP: Network Scanning
	+ TTP: Credential Theft
* **Data Exfiltration**: Attackers encrypt sensitive data and demand payme

In [15]:
from langchain_community.document_loaders import TextLoader

documents = []

for file in os.listdir(data_path):
    if file.endswith(".txt"):
        loader = TextLoader(os.path.join(data_path, file))
        documents.extend(loader.load())

print("Documents loaded:", len(documents))

Documents loaded: 5


In [21]:
query = "Explain phishing attacks"

results = multi_vector_db.similarity_search_with_score(query, k=5)

for doc, score in results:
    print("\nSCORE:", score)
    print(doc.metadata)
    print(doc.page_content)


SCORE: 0.44725799560546875
{'source': 'C:\\Users\\Abhishek N Salian\\Desktop\\cyber-ai-platform\\data\\threat_intel\\phishing.txt'}
Phishing is a social engineering attack where attackers trick users into revealing credentials or sensitive information.

Phishing commonly uses:
- fake emails
- spoofed websites
- malicious links
- fake login pages

SCORE: 0.44725799560546875
{'source': 'C:\\Users\\Abhishek N Salian\\Desktop\\cyber-ai-platform\\data\\threat_intel\\phishing.txt'}
Phishing is a social engineering attack where attackers trick users into revealing credentials or sensitive information.

Phishing commonly uses:
- fake emails
- spoofed websites
- malicious links
- fake login pages

SCORE: 0.6683594584465027
{'source': 'C:\\Users\\Abhishek N Salian\\Desktop\\cyber-ai-platform\\data\\threat_intel\\phishing.txt'}
Phishing is a cyberattack technique in which attackers impersonate trusted entities to steal sensitive information.

Common phishing techniques:
- email spoofing
- fake l